In [1]:
# !conda activate fakenews
!pip install numpy --force-reinstall

  Using cached numpy-2.3.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.3.1-cp311-cp311-win_amd64.whl (13.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.0
    Uninstalling numpy-2.2.0:
      Successfully uninstalled numpy-2.2.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.3.1 which is incompatible.
torchvision 0.16.0+cu118 requires torch==2.1.0+cu118, but you have torch 2.7.0 which is incompatible.


In [1]:
import numpy as np
print(np.__version__)

2.2.0


In [ ]:
import logging
import torch

from transformers import (
    BlipProcessor, BlipForConditionalGeneration,
    Wav2Vec2Processor, Wav2Vec2ForSequenceClassification,
    TrOCRProcessor, VisionEncoderDecoderModel,
    ViTFeatureExtractor, ViTModel,
    CLIPProcessor, CLIPModel,
    pipeline
)

device = "cuda" if torch.cuda.is_available() else "cpu"

def load_clip():
    logging.info("Carregando modelo de transcrição e sumarização de vídeo...")
    return (
        CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device),
        CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)
    )

In [ ]:
from ollama import Client

api_host = ""
client = Client(host=api_host)

# Extrair e exibir apenas os nomes dos modelos
models = client.list()
model_names = [model['model'] for model in models['models']]

model_names

['nomic-embed-text:latest',
 'llama3.2-vision:11b',
 'qwen2.5vl:7b',
 'gemma3:12b',
 'deepseek-r1:8b',
 'llama3.1:8b',
 'llama3.2:3b',
 'phi3:14b',
 'qwen2.5:14b',
 'qwen2.5:7b',
 'deepseek-r1:32b',
 'gemma2:9b',
 'qwen2:7b']

In [4]:
import logging
import torch

from transformers import (
    BlipProcessor, BlipForConditionalGeneration,
    Wav2Vec2Processor, Wav2Vec2ForSequenceClassification,
    TrOCRProcessor, VisionEncoderDecoderModel,
    ViTFeatureExtractor, ViTModel,
    CLIPProcessor, CLIPModel,
    pipeline
)

device = "cuda" if torch.cuda.is_available() else "cpu"

def load_clip():
    logging.info("Carregando modelo de transcrição e sumarização de vídeo...")
    return CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device),CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [7]:
!pip install pydub

  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pydub-0.25.1-py2.py3-none-any.whl (32 kB)


In [ ]:
!pip install opencv-python

In [5]:
import os
import re
# import cv2
import time
import json
import base64
import shutil
import random
import requests
import whisper
import numpy as np
import pandas as pd
import moviepy.editor as mp
import cv2

from pathlib import Path
from collections import Counter

from whisper import load_model
from pydub import AudioSegment
from pydub.utils import mediainfo
# from resemblyzer import VoiceEncoder, preprocess_wav
from scipy.spatial.distance import cosine
# from spectralcluster import SpectralClusterer
from pyannote.audio import Pipeline
from ollama import Client, ResponseError
from openai import OpenAI

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# carregnado modelo de Clip embeddings
clip_model, clip_processor = load_clip()

# Lista de emoções
emotions = ["feliz", "neutro", "triste", "raiva", "medo", "surpreso", "calmo", "tédio", "cansado", "irritado"]

api_host = ""
client = Client(host=api_host)

def generate_correlation_matrix(synced_data):
    if not synced_data:
        raise ValueError("Erro: synced_data está vazio ou None!")

    # Verifica se synced_data é uma lista de tuplas do tamanho correto
    if not isinstance(synced_data, list) or not all(
        isinstance(row, tuple) and len(row) == 4 for row in synced_data
    ):
        raise ValueError(f"Formato inválido de synced_data: {synced_data}")

    return pd.DataFrame(
        synced_data,
        columns=[
            "Timestamp",
            "Frame",
            "Transcription",
            "OCR_Text",
        ],
    )


# Função para extração de texto (OCR)
def extract_text(image_path, reader):
    """
    Extrai texto de uma imagem usando OCR.
    
    :param image_path: Caminho da imagem a ser processada.
    :param reader: Modelo OCR para leitura de texto.
    :return: Texto extraído da imagem.
    """
    results = reader.readtext(image_path, detail=0)
    return " ".join(results) if results else ""

# Função para pré-processamento da imagem
def preprocess_image(image_path):
    """
    Converte uma imagem para escala de cinza e aplica equalização de histograma.
    
    :param image_path: Caminho da imagem.
    :return: Imagem processada em escala de cinza com limiarização adaptativa.
    """
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    processed = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C_,
                                      cv2.THRESH_BINARY, 11, 2)
    return processed


# Função para detectar emoção no áudio
def detect_emotion_from_audio(audio_array, sr, audio_processor, audio_model):
    """Detecta a emoção diretamente de um array de áudio sem salvar no disco.
    
    :param audio_array: Array numpy contendo os dados de áudio.
    :param sr: Taxa de amostragem do áudio.
    :param audio_processor: Processador de áudio para transformar os dados.
    :param audio_model: Modelo de IA para detecção de emoções.
    :return: Emoção detectada e sua confiança.
    """
    input_values = audio_processor(audio_array, sampling_rate=sr, return_tensors="pt", padding=True).input_values.to(device)
    with torch.no_grad():
        logits = audio_model(input_values).logits
    scores = torch.nn.functional.softmax(logits, dim=-1)
    emotion_idx = scores.argmax().item()
    emotion = emotions[emotion_idx]
    confidence = scores[0, emotion_idx].item()
    return emotion, confidence

def tem_repeticao_excessiva(texto, limite=5, n=4):
    if not isinstance(texto, str):
        return False

    palavras = texto.strip().lower().split()
    if len(palavras) < n:
        return False

    ngramas = [' '.join(palavras[i:i+n]) for i in range(len(palavras) - n + 1)]

    ultima = None
    repeticoes = 1
    for ng in ngramas:
        if ng == ultima:
            repeticoes += 1
            if repeticoes >= limite:
                return True
        else:
            repeticoes = 1
        ultima = ng
    return False


def transcribe_and_diarize(video_path, hf_token, model_type="large", granularity=0.5, max_tentativas=5):
    if shutil.which("ffmpeg") is None:
        raise EnvironmentError("FFmpeg não foi encontrado. Instale e adicione ao PATH.")

    # Extrair áudio
    audio_path = "temp_audio.wav"
    video = mp.VideoFileClip(video_path)
    video.audio.write_audiofile(audio_path, verbose=False, logger=None)
    info = mediainfo(audio_path)

    # Carrega modelo Whisper
    model = whisper.load_model(model_type)

    tentativa = 0
    result = None

    while tentativa < max_tentativas:
        tentativa += 1
        # print(f"Tentativa {tentativa} de transcrição...")

        # Varia parâmetros de acordo com a tentativa
        temperature = 0.0 + 0.2 * (tentativa - 1)  # aumenta aleatoriedade
        compression_ratio_threshold = 2.4 - 0.3 * (tentativa - 1)  # tolerância a repetição
        logprob_threshold = -1.0 + 0.2 * (tentativa - 1)
        beam_size = 5 if tentativa >= 3 else None  # ativa beam search após 2 tentativas

        result = model.transcribe(
            audio_path,
            language="pt",
            fp16=False,
            word_timestamps=True,
            temperature=temperature,
            compression_ratio_threshold=compression_ratio_threshold,
            logprob_threshold=logprob_threshold,
            no_speech_threshold=0.0,
            beam_size=beam_size,
        )

        texto = result.get("text", "").strip()
        if not tem_repeticao_excessiva(texto, limite=5, n=4) and len(texto.split()) > 5:
            # print("✅ Transcrição válida.")
            break
        else:
            print("⚠️ Resultado com repetições ou baixa qualidade. Tentando com novos parâmetros...")

    whisper_segments = result.get("segments", [])

    # Se não houver segmentos, retorna um padrão
    if not whisper_segments:
        os.remove(audio_path)
        return pd.DataFrame([{
            "start": 0.0,
            "end": float(info["duration"]),
            "text": "Não possui áudio",
            "speaker": "Unknown"
        }])

    # 3. Diarização com pyannote
    try:
        pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", use_auth_token=hf_token)
        diarization = pipeline(audio_path)

        # 4. Dividir segmentos em janelas pequenas
        diarized_windows = []
        for turn, _, speaker in diarization.itertracks(yield_label=True):
            t = turn.start
            while t < turn.end:
                t_end = min(t + granularity, turn.end)
                diarized_windows.append({
                    "start": round(t, 2),
                    "end": round(t_end, 2),
                    "speaker": speaker
                })
                t = t_end
        df_speakers = pd.DataFrame(diarized_windows)

        # Garantir colunas obrigatórias
        if df_speakers.empty or not all(c in df_speakers.columns for c in ["start", "end", "speaker"]):
            df_speakers = None

    except Exception as e:
        # print(f"⚠️ Erro na diarização: {e}")
        df_speakers = None

    # 5. Cruzar com a transcrição
    annotated = []
    for seg in whisper_segments:
        text_start = seg["start"]
        text_end = seg["end"]
        text = seg["text"]

        if df_speakers is not None:
            matching = df_speakers[
                (df_speakers["start"] < text_end) & (df_speakers["end"] > text_start)
            ]
            speaker = matching["speaker"].mode().iloc[0] if not matching.empty else "Unknown"
        else:
            speaker = "Unknown"

        annotated.append({
            "start": text_start,
            "end": text_end,
            "text": text,
            "speaker": speaker
        })

    os.remove(audio_path)
    return pd.DataFrame(annotated)


def get_clip_embedding(frame):
    inputs = clip_processor(images=[frame], return_tensors="pt").to(device)
    with torch.no_grad():
        embedding = clip_model.get_image_features(**inputs)
    return embedding.cpu().numpy().flatten()


def summarize_video_frames(video_path, frame_interval=1, threshold=0.2, window_size=5):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    current_frame_index = 0
    saved_frames = []

    # Carrega o primeiro frame como referência
    cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame_index)
    ret, reference_frame = cap.read()
    if not ret:
        cap.release()
        return []

    reference_embedding = get_clip_embedding(reference_frame)
    saved_frames.append((0.0, reference_frame))

    while current_frame_index < total_frames:
        change_detected = False

        for offset in range(1, window_size + 1):
            next_index = current_frame_index + offset * int(fps * frame_interval)
            if next_index >= total_frames:
                break

            cap.set(cv2.CAP_PROP_POS_FRAMES, next_index)
            ret, next_frame = cap.read()
            if not ret:
                continue

            next_embedding = get_clip_embedding(next_frame)
            distance = cosine(reference_embedding, next_embedding)

            if distance > threshold:
                timestamp = next_index / fps
                saved_frames.append((timestamp, next_frame))
                reference_embedding = next_embedding
                current_frame_index = next_index
                change_detected = True
                break

        if not change_detected:
            current_frame_index += window_size * int(fps * frame_interval)

    cap.release()
    return saved_frames


def save_summarized_frames(
    video_path,
    output_folder="frames_summary",
    frame_interval=1,
    threshold=0.25,
    window_size=5,
    limit=45
):
    os.makedirs(output_folder, exist_ok=True)
    
    # Força leitura do primeiro frame
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    success, first_frame = cap.read()
    saved_paths = []

    if success:
        first_frame_path = os.path.join(output_folder, "scene_0.00.jpg")
        cv2.imwrite(first_frame_path, first_frame)
        saved_paths.append((0.0, first_frame_path))

    # Demais frames resumidos por mudança visual
    summarized_frames = summarize_video_frames(video_path, frame_interval, threshold, window_size)

    for i, (timestamp, frame) in enumerate(summarized_frames):
        # Pula timestamp 0.0 se já foi salvo manualmente
        if abs(timestamp - 0.0) < 1e-2:
            continue

        frame_path = os.path.join(output_folder, f"scene_{timestamp:.2f}.jpg")
        cv2.imwrite(frame_path, frame)
        saved_paths.append((timestamp, frame_path))

        # Verifica necessidade de inserção entre frames
        if i < len(summarized_frames) - 1:
            next_timestamp = summarized_frames[i + 1][0]
            gap = next_timestamp - timestamp

            if gap > limit:
                num_inserts = int(gap // limit)
                for j in range(1, num_inserts + 1):
                    new_ts = timestamp + j * limit
                    frame_number = int(new_ts * fps)
                    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
                    ret, intermediate_frame = cap.read()
                    if ret:
                        intermediate_path = os.path.join(output_folder, f"scene_{new_ts:.2f}.jpg")
                        cv2.imwrite(intermediate_path, intermediate_frame)
                        saved_paths.append((new_ts, intermediate_path))

    cap.release()
    saved_paths = sorted(saved_paths, key=lambda x: x[0])
    return saved_paths


def describe_image(
    image_path,
    model="qwen2.5vl:7b",
    temperature=0.1,
    max_retries=5,
    wait_seconds=5,
):
    total_start = time.time()
    # print(f"\n⏱️ Início da descrição da imagem: {image_path}")

    # 1. Codifica imagem
    start = time.time()
    with open(image_path, "rb") as img_file:
        img_base64 = base64.b64encode(img_file.read()).decode("utf-8")
    # print(f"✅ Imagem codificada em {time.time() - start:.2f} segundos")

    # 2. Prompt detalhado
    prompt = (
        "Descreva objetivamente o que está visível na imagem, com atenção a textos presentes "
        "(como placas, legendas, logotipos) e tente inferir o contexto (TV, rede social, evento). "
        "Informe também um grau de confiança de 0 a 1. Responda com texto direto."
    )

    # 3. Tenta gerar com retries
    for attempt in range(1, max_retries + 1):
        try:
            start = time.time()
            response = client.generate(
                model=model,
                prompt=prompt,
                images=[img_base64],
                options={"temperature": temperature}
            )
            # print(f"📡 Resposta gerada em {time.time() - start:.2f} segundos")
            break  # Sucesso
        except ResponseError as e:
            # print(f"⚠️ Tentativa {attempt}/{max_retries} falhou: {e}")
            if attempt == max_retries:
                raise RuntimeError(f"Erro após {max_retries} tentativas: {e}")
            time.sleep(wait_seconds)

    # 4. Processa resposta
    raw_text = response.get("response", "").strip()
    confidence_match = re.search(r"\b(0(?:\.\d+)?|1(?:\.0+)?)\b", raw_text)
    confidence = float(confidence_match.group(1)) if confidence_match else None

    # print(f"✅ Concluído em {time.time() - total_start:.2f} segundos")

    return {
        "response": raw_text,
        "confidence": confidence
    }

def match_frames_to_transcription(transcription_segments_df, frame_list):
    """
    Associa cada frame ao segmento de transcrição mais próximo.
    Evita repetição de 'text' para mesmos intervalos (start, end).

    :param transcription_segments_df: DataFrame com ['start', 'end', 'text', 'speaker']
    :param frame_list: Lista de tuplas (timestamp, frame_path)
    :return: DataFrame com colunas: ['speaker', 'text', 'start', 'end', 'image_description', 'confidence']
    """
    if transcription_segments_df.empty or not frame_list:
        raise ValueError("Entrada vazia ou inválida.")

    frame_list = sorted(frame_list, key=lambda x: x[0])
    data_final = []
    transcricoes_usadas = set()

    for timestamp, frame_path in frame_list:
        result = describe_image(frame_path)  # retorna {'response': ..., 'confidence': ...}

        # Encontra linha mais próxima
        idx_mais_proximo = (transcription_segments_df['start'] - timestamp).abs().argsort().iloc[0]
        closest_row = transcription_segments_df.iloc[idx_mais_proximo]

        # Cria chave única pelo intervalo de tempo
        chave_transcricao = (closest_row['start'], closest_row['end'], closest_row['text'])

        # Garante que text apareça só uma vez por (start, end, text)
        if chave_transcricao in transcricoes_usadas:
            texto = ''
        else:
            texto = closest_row['text']
            transcricoes_usadas.add(chave_transcricao)

        linha = {
            'speaker': closest_row['speaker'],
            'text': texto,
            'start': closest_row['start'],
            'end': closest_row['end'],
            'image_description': result.get("response", ""),
            'confidence': result.get("confidence", None)
        }

        data_final.append(linha)

    df = pd.DataFrame(data_final)

    # Agrupa segmentos com mesma imagem, confiança e speaker consecutivos
    df['group_shift'] = (
        (df['speaker'] != df['speaker'].shift()) |
        (df['image_description'] != df['image_description'].shift()) |
        (df['confidence'] != df['confidence'].shift())
    ).cumsum()

    df_grouped = df.groupby('group_shift').agg({
        'speaker': 'first',
        'text': lambda x: ' '.join(filter(None, x)),  # junta apenas textos não vazios
        'start': 'first',
        'end': 'last',
        'image_description': 'first',
        'confidence': 'first'
    }).reset_index(drop=True)

    return df_grouped


def _query_model(task, text, model, client=None, client_openai=None):
    """
    Consulta modelo OpenAI ou Ollama para classificação de emoção.
    """
    prompt = task + "\n===\n" + text

    if model.startswith("gpt"):
        response = client_openai.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": ""},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0
        )
        return response.choices[0].message.content.strip().lower()

    else:
        if client is None:
            raise ValueError("Client Ollama necessário para modelos não OpenAI.")
        response = client.generate(
            model=model,
            prompt=prompt,
            options={"temperature": 0.0}
        )
        return response["response"].strip().lower()


def emotion_classification_comite(synced_data):
    """
    Aplica classificação emocional com comitê de LLMs e adiciona colunas ao synced_data:
    ['emotion', 'certeza_emocional', 'respostas_modelos']
    
    :param synced_data: DataFrame com colunas ['speaker', 'text', ...]
    :param client: Instância de ollama.Client
    :return: synced_data (DataFrame com emoções anexadas)
    """
    api_key = ""

    client_openai = OpenAI(api_key=api_key)

    models = ["gpt-4.1-mini", "llama3.2:3b", "qwen2:7b"]
    emotions = ["feliz", "neutro", "triste", "raiva"]
    resultados = []

    frases_unicas = synced_data[["speaker", "text"]].drop_duplicates()
    task = (
        "Classifique a emoção principal expressa no seguinte texto. "
        "Responda apenas com uma das opções: feliz, neutro, triste ou raiva."
    )

    for _, row in frases_unicas.iterrows():
        speaker = row["speaker"]
        text = row["text"]
        respostas = []

        for model in models:
            try:
                resposta = _query_model(task, text, model, client=client, client_openai=client_openai)
                for emo in emotions:
                    if emo in resposta:
                        respostas.append(emo)
                        break
            except Exception as e:
                # print(f"⚠️ Erro com o modelo {model}: {e}")
                respostas.append("erro")

        respostas_validas = [r for r in respostas if r in emotions]

        if not respostas_validas:
            pseudolabel = "neutro"
            certeza = "baixa"
        else:
            contagem = Counter(respostas_validas)
            pseudolabel, votos = contagem.most_common(1)[0]
            if votos == 3:
                certeza = "alta"
            elif votos == 2:
                certeza = "média"
            else:
                certeza = "baixa"

        resultados.append({
            "speaker": speaker,
            "text": text,
            "emotion": pseudolabel,
            "emocional_confidence": certeza,
            "answer_gpt_l_q": respostas
        })

    df_emocoes = pd.DataFrame(resultados)

    # Merge com synced_data original
    synced_data_final = synced_data.merge(
        df_emocoes,
        on=["speaker", "text"],
        how="left"
    )

    return synced_data_final

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
import time
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

input_dir = 'dataset'
output_root = 'working_v2'
# whisper_models = ["tiny", "large", "large-v3"]
whisper_models = ["large-v3"]
hf_token = ''

for model_type in whisper_models:
    all_dfs = []

    out_path = f"{output_root}/results_whisper_{model_type}.csv"
    processed_ids = set()

    # Verifica se já existe arquivo de saída
    if os.path.exists(out_path):
        try:
            df_existing = pd.read_csv(out_path)
            # Remove extensão, se houver, ao comparar
            processed_ids = set(Path(v).stem for v in df_existing["video_id"].astype(str).unique())
            print(f"🔁 {len(processed_ids)} vídeos já processados encontrados em {out_path}.")
        except Exception as e:
            print(f"⚠️ Não foi possível ler {out_path}. Iniciando novo.")
            df_existing = None

    for filename in os.listdir(input_dir):
        if not filename.endswith(".mp4"):
            continue

        # if filename not in ['186.mp4', '125.mp4']:
        #     continue

        file_path = os.path.join(input_dir, filename)
        video_id = Path(filename).stem
        if video_id in processed_ids:
            print(f"⏩ Pulando vídeo já processado: {video_id}")
            continue

        video_id = Path(filename).stem
        start_time = time.time()

        try:
            # === Remover pasta de frames anterior (se existir) ===
            frame_output_folder = f"{output_root}/{video_id}/{model_type}/frames"
            if os.path.exists(frame_output_folder):
                shutil.rmtree(frame_output_folder)
            os.makedirs(frame_output_folder, exist_ok=True)

            # === 1. Transcrição e diarização ===
            transcription_segments = transcribe_and_diarize(
                file_path, hf_token=hf_token, model_type=model_type
            )

            # === 2. Extração de frames ===
            frame_list = save_summarized_frames(
                file_path, output_folder=frame_output_folder, frame_interval=2
            )

            # print(transcription_segments, frame_list)

            # === 3. Sincronização ===
            synced_data = match_frames_to_transcription(transcription_segments, frame_list)

            # === 4. Classificação emocional ===
            synced_data = emotion_classification_comite(synced_data)

            # === 5. Metadados + salvar ===
            processing_duration = time.time() - start_time
            synced_data["video_id"] = video_id
            synced_data["whisper_model"] = model_type
            synced_data["processing_time"] = processing_duration
            all_dfs.append(synced_data)
            print(f"✅ Concluído: {video_id} com {model_type}")

            # === Limpar arquivo de áudio temporário ===
            temp_audio_path = "temp_audio.wav"
            if os.path.exists(temp_audio_path):
                os.remove(temp_audio_path)

        except Exception as e:
            print(f"\n❌ Erro ao processar vídeo: {video_id}, modelo: {model_type}")
            import traceback
            traceback.print_exc()
            raise e

    # === Salvar ===
    if all_dfs:
        df_resultado = pd.concat(all_dfs, ignore_index=True)
        if os.path.exists(out_path):
            df_existente = pd.read_csv(out_path)
            df_resultado = pd.concat([df_existente, df_resultado], ignore_index=True)
        df_resultado.to_csv(out_path, index=False)
        print(f"💾 Resultados atualizados salvos em: {out_path}")
    else:
        print(f"⚠️ Nenhum dado novo processado com sucesso para modelo: {model_type}")

DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _recover


✅ Concluído: 1 com large-v3
✅ Concluído: 10 com large-v3
✅ Concluído: 105 com large-v3
⚠️ Resultado com repetições ou baixa qualidade. Tentando com novos parâmetros...
⚠️ Resultado com repetições ou baixa qualidade. Tentando com novos parâmetros...
✅ Concluído: 11 com large-v3
✅ Concluído: 111 com large-v3
✅ Concluído: 113 com large-v3
✅ Concluído: 115 com large-v3
✅ Concluído: 117 com large-v3
✅ Concluído: 12 com large-v3
✅ Concluído: 125 com large-v3
✅ Concluído: 126 com large-v3
✅ Concluído: 127 com large-v3
✅ Concluído: 128 com large-v3
✅ Concluído: 129 com large-v3
✅ Concluído: 13 com large-v3
✅ Concluído: 130 com large-v3
✅ Concluído: 131 com large-v3
✅ Concluído: 132 com large-v3
✅ Concluído: 133 com large-v3
✅ Concluído: 134 com large-v3
✅ Concluído: 135 com large-v3
✅ Concluído: 137 com large-v3
✅ Concluído: 138 com large-v3
✅ Concluído: 139 com large-v3
✅ Concluído: 14 com large-v3
✅ Concluído: 140 com large-v3
✅ Concluído: 141 com large-v3
✅ Concluído: 142 com large-v3
✅ Con

In [ ]:
result = pd.read_csv(r"working_v2\results_whisper_large-v3.csv")
result.shape

(665, 12)

In [63]:
df_resultado

,speaker,text,start,end,image_description,confidence,emotion,emocional_confidence,answer_gpt_l_q,video_id,whisper_model,processing_time
0,SPEAKER_00,Nós já começamos a reconstruir 186 milhões de...,0.0,6.0,A imagem mostra uma conversa entre duas pessoa...,0.9,neutro,média,"[neutro, feliz, neutro]",125,large-v3,70.917654
1,SPEAKER_00,,0.0,6.0,"A imagem mostra uma pessoa em um palco, vestin...",0.9,neutro,baixa,"[neutro, feliz]",125,large-v3,70.917654
2,SPEAKER_00,,0.0,6.0,A imagem é uma captura de tela de um artigo de...,1.0,neutro,baixa,"[neutro, feliz]",125,large-v3,70.917654
3,SPEAKER_00,Se você vai no supermercado e você desconfia ...,0.0,5.6,"A imagem é completamente preta, sem nenhum tex...",0.0,neutro,média,"[neutro, feliz, neutro]",186,large-v3,53.337834
4,SPEAKER_00,,0.0,5.6,A imagem mostra uma cena com um homem de costa...,0.8,neutro,baixa,"[neutro, feliz]",186,large-v3,53.337834


In [ ]:
df_resultado = pd.concat(all_dfs, ignore_index=True)
if os.path.exists(out_path):
    df_existente = pd.read_csv(out_path)
    df_resultado = pd.concat([df_existente, df_resultado], ignore_index=True)
df_resultado.to_csv(out_path, index=False)
print(f"💾 Resultados atualizados salvos em: {out_path}")

💾 Resultados atualizados salvos em: working/results_whisper_large.csv
